# SWE-Pocket — Reproduction Notebook

Reproduces every quantitative claim in *SWE-Pocket: A Size-Stratified Evaluation
Resource for Local Coding Agents* without a GPU or a model run.

Inputs (released in this repo):
- `data/joined_structure_gold.csv` — structure x gold-patch x test-set, 129 rows
- `data/compact_joined.json` — the same join for the 24 compact tasks
- `resources/strata_table.json` — the shipped summary table
- `resources/compact_tasks.json` — the 24 mined tasks and their validation flags

The competition's `tasks.jsonl` itself is not redistributed; the derived join
tables above are what this notebook reads.

## 0. Setup

In [1]:
import os, csv, json, statistics
from collections import Counter

# Resolve repo root whether the notebook is run from notebooks/ or the root
ROOT = os.getcwd() if os.path.isdir("data") else os.path.abspath("..")
P = lambda *p: os.path.join(ROOT, *p)

LARGE = list(csv.DictReader(open(P("data/joined_structure_gold.csv"))))
COMPACT = json.load(open(P("data/compact_joined.json")))
print("large tasks:", len(LARGE), "| compact tasks:", len(COMPACT),
      "| total:", len(LARGE) + len(COMPACT))

large tasks: 129 | compact tasks: 24 | total: 153


## 1. The resource — 129 provided + 24 mined = 153 tasks
Per-repo counts, as reported in §3.

In [2]:
print("Large stratum (provided):", dict(Counter(r["repo"] for r in LARGE)))
print("Compact stratum (mined):  ", dict(Counter(r["repo"] for r in COMPACT)))
assert len(LARGE) == 129 and len(COMPACT) == 24

Large stratum (provided): {'fastapi': 67, 'requests': 13, 'rich': 48, 'httpx': 1}
Compact stratum (mined):   {'packaging': 6, 'prettytable': 6, 'parso': 6, 'python-dotenv': 6}


### 1.1 Task validation
Every compact task was replayed from source on 2026-11-10 (see `VALIDATION.md`):
apply the test patch at the base commit, run the affected tests, then apply the
gold patch and run them again. `discriminates` records whether the task fails
before the gold patch and passes after.

In [3]:
TASKS = {t["instance_id"]: t for t in json.load(open(P("resources/compact_tasks.json")))}
validated = [t for t in COMPACT if TASKS[t["instance_id"]].get("validated")]
disc = [t for t in COMPACT if TASKS[t["instance_id"]].get("discriminates")]
print(f"compact tasks replayed: {len(validated)}/{len(COMPACT)}")
print(f"discriminating (base-fail, gold-pass): {len(disc)}/{len(COMPACT)}")
print("non-discriminating:", [t["instance_id"] for t in COMPACT
                               if not TASKS[t["instance_id"]].get("discriminates")])

compact tasks replayed: 24/24
discriminating (base-fail, gold-pass): 21/24
non-discriminating: ['parso_241', 'parso_233', 'parso_227']


## 2. Table 1 — repository structure by stratum
Means per repository; churn is the median gold-patch churn. `connected` is the
mean count of nodes with at least one call edge. This is the bias result (§4.2):
raw node count is not the same thing as navigable structure.

In [4]:
def m(rows, k): return statistics.mean(float(r[k]) for r in rows)
def med(rows, k): return statistics.median(float(r[k]) for r in rows)

def connected(r):
    return float(r["nodes"]) * (1.0 - float(r["isolated_ratio"]))

def row(repo, stratum, rows):
    return dict(repo=repo, stratum=stratum, n=len(rows),
                nodes=round(m(rows, "nodes")), edges=round(m(rows, "edges")),
                isolated=round(100 * m(rows, "isolated_ratio"), 1),
                connected=round(statistics.mean(connected(r) for r in rows)),
                density=round(m(rows, "density"), 2),
                max_depth=round(m(rows, "max_depth")),
                churn=round(med(rows, "gold_churn_total")))

table = []
for repo in ["fastapi", "rich", "requests", "httpx"]:
    table.append(row(repo, "large", [r for r in LARGE if r["repo"] == repo]))
for repo in ["packaging", "parso", "prettytable", "python-dotenv"]:
    table.append(row(repo, "compact", [r for r in COMPACT if r["repo"] == repo]))

hdr = (f"{'repo':16}{'stratum':9}{'n':>4}{'raw':>7}{'isolated%':>11}"
       f"{'connected':>11}{'edges':>8}{'density':>9}{'depth':>7}{'churn':>7}")
print(hdr); print("-" * len(hdr))
for t in table:
    print(f"{t['repo']:16}{t['stratum']:9}{t['n']:>4}{t['nodes']:>7}{t['isolated']:>11}"
          f"{t['connected']:>11}{t['edges']:>8}{t['density']:>9}{t['max_depth']:>7}{t['churn']:>7}")

repo            stratum     n    raw  isolated%  connected   edges  density  depth  churn
-----------------------------------------------------------------------------------------
fastapi         large      67   3968       68.7       1235    2458     0.62     57     19
rich            large      48   1970        3.5       1902    5626     2.86    172     10
requests        large      13    733        7.9        675    1819     2.48     53      5
httpx           large       1    740        2.8        719    1618     2.19     33     86
packaging       compact     6    579       56.8        250     569     0.98      9     12
parso           compact     6    604       75.7        147     232     0.38      5      6
prettytable     compact     6    215       68.0         69      29     0.14      2     12
python-dotenv   compact     6     75       41.1         44      82     1.08      6     19


### 2.1 Size ranges (the stratification is genuine)
Raw: large 711–4,597 vs compact 75–606. Connected: 654–1,952 vs 44–250.

In [5]:
ln = [int(r["nodes"]) for r in LARGE]
cn = [int(r["nodes"]) for r in COMPACT]
lc = [connected(r) for r in LARGE]
cc = [connected(r) for r in COMPACT]
print(f"large   raw: {min(ln)}–{max(ln)}   connected: {min(lc):.0f}–{max(lc):.0f}")
print(f"compact raw: {min(cn)}–{max(cn)}   connected: {min(cc):.0f}–{max(cc):.0f}")
assert max(cn) < min(ln) and max(cc) < min(lc), "strata must be disjoint in size"

large   raw: 711–4597   connected: 654–1952
compact raw: 75–606   connected: 44–250


### 2.2 Repository-level rank agreement
Spearman correlation across the eight repository means (descriptive, n = 8).
The paper reports three: raw nodes vs depth, connected nodes vs depth, and
connected nodes vs density.

In [6]:
def spearman(xs, ys):
    def rank(v):
        order = sorted(range(len(v)), key=lambda i: v[i]); r = [0.0] * len(v); i = 0
        while i < len(v):
            j = i
            while j + 1 < len(v) and v[order[j + 1]] == v[order[i]]: j += 1
            for k in range(i, j + 1): r[order[k]] = (i + j) / 2 + 1
            i = j + 1
        return r
    rx, ry = rank(xs), rank(ys)
    d2 = sum((a - b) ** 2 for a, b in zip(rx, ry)); n = len(xs)
    return 1 - 6 * d2 / (n * (n * n - 1))

means = {t["repo"]: t for t in table}
col = lambda k: [means[t["repo"]][k] for t in table]
print(f"raw nodes vs depth     rho = {spearman(col('nodes'), col('max_depth')):+.2f}")
print(f"connected vs depth     rho = {spearman(col('connected'), col('max_depth')):+.2f}")
print(f"connected vs density   rho = {spearman(col('connected'), col('density')):+.2f}")

raw nodes vs depth     rho = +0.83
connected vs depth     rho = +0.90
connected vs density   rho = +0.55


## 3. Table 2 — structure does not predict fix size (§4.3)
Pearson correlation of each structural dimension with gold-patch churn: pooled
over 129 tasks and within each repository. Tasks cluster in four repositories,
so the pooled n = 129 overstates precision; the per-repository columns are the
primary evidence.

In [7]:
import math
def pearson(xs, ys):
    n = len(xs); mx = sum(xs)/n; my = sum(ys)/n
    cov = sum((a-mx)*(b-my) for a, b in zip(xs, ys))
    vx = sum((a-mx)**2 for a in xs); vy = sum((b-my)**2 for b in ys)
    return cov / math.sqrt(vx*vy) if vx*vy else 0.0

def corr(rows, axis):
    return pearson([float(r[axis]) for r in rows],
                   [float(r["gold_churn_total"]) for r in rows])

AXES = ["nodes", "edges", "density", "max_depth"]
REPOS = ["fastapi", "rich", "requests"]
hdr = f"{'axis':10}{'pooled':>9}" + "".join(f"{p:>10}" for p in REPOS)
print(hdr); print("-" * len(hdr))
for axis in AXES:
    line = f"{axis:10}{corr(LARGE, axis):>+9.3f}"
    for p in REPOS:
        line += f"{corr([r for r in LARGE if r['repo'] == p], axis):>+10.3f}"
    print(line)

axis         pooled   fastapi      rich  requests
-------------------------------------------------
nodes        -0.014    +0.228    +0.077    +0.251
edges        +0.088    +0.111    +0.082    +0.235
density      +0.055    -0.172    +0.118    +0.119
max_depth    +0.102    -0.026    +0.163    +0.653


### 3.1 The typical task is small and single-file

In [8]:
sf = [r for r in LARGE if int(r["gold_files"]) == 1]
churn = [int(r["gold_churn_total"]) for r in LARGE]
print(f"single-source-file tasks: {len(sf)}/{len(LARGE)} = {len(sf)/len(LARGE):.0%}")
print(f"median gold churn (all):        {statistics.median(churn):.0f} lines")
print(f"median gold churn (single-file): {statistics.median(int(r['gold_churn_total']) for r in sf):.0f} lines")

single-source-file tasks: 91/129 = 71%
median gold churn (all):        13 lines
median gold churn (single-file): 7 lines


### 3.2 Why the pooled number is not the story
The pooled node correlation is negative, but the sign within each repository is
positive: composition, not independence. This cell also lists the outliers.

In [9]:
def centred_corr(axis):
    xs, ys = [], []
    for repo in REPOS:
        rs = [r for r in LARGE if r["repo"] == repo]
        mx = statistics.mean(float(r[axis]) for r in rs)
        my = statistics.mean(float(r["gold_churn_total"]) for r in rs)
        xs += [float(r[axis]) - mx for r in rs]
        ys += [float(r["gold_churn_total"]) - my for r in rs]
    return pearson(xs, ys)

print(f"pooled nodes  r = {corr(LARGE, 'nodes'):+.3f}")
print(f"centred nodes r = {centred_corr('nodes'):+.3f}")
for p in REPOS:
    rs = [r for r in LARGE if r["repo"] == p]
    print(f"  {p:10} r = {corr(rs, 'nodes'):+.3f}  (n = {len(rs)})")
out = sorted(LARGE, key=lambda r: -int(r["gold_churn_total"]))[:3]
print("largest fixes (outliers):")
for r in out:
    print(f"  {r['instance_id']:18} {r['gold_churn_total']:>6} lines, {r['gold_files']} files")

pooled nodes  r = -0.014
centred nodes r = +0.050
  fastapi    r = +0.228  (n = 67)
  rich       r = +0.077  (n = 48)
  requests   r = +0.251  (n = 13)
largest fixes (outliers):
  rich_3930           12740 lines, 26 files
  fastapi_14609        2067 lines, 20 files
  fastapi_15745        1234 lines, 4 files


## 4. Test sets recovered from provided `test_patch` (§3.3)
FAIL_TO_PASS / PASS_TO_PASS totals.

In [10]:
f2p = sum(int(r["f2p_count"]) for r in LARGE)
p2p = sum(int(r["p2p_count"]) for r in LARGE)
with_f2p = sum(1 for r in LARGE if int(r["f2p_count"]) > 0)
print(f"tasks with >=1 FAIL_TO_PASS: {with_f2p}/{len(LARGE)}")
print(f"total FAIL_TO_PASS functions: {f2p}")
print(f"total PASS_TO_PASS functions: {p2p}")

tasks with >=1 FAIL_TO_PASS: 106/129
total FAIL_TO_PASS functions: 464
total PASS_TO_PASS functions: 124


## 5. Cross-validate the shipped stratum table
Confirms `resources/strata_table.json` matches this notebook's computation.

In [11]:
shipped = {t["repo"]: t for t in json.load(open(P("resources/strata_table.json")))}
recomputed = {t["repo"]: t for t in table}
for repo, s in shipped.items():
    r = recomputed[repo]
    assert (s["nodes"], s["edges"], s["max_depth"], s["median_churn"]) == \
           (r["nodes"], r["edges"], r["max_depth"], r["churn"]), (repo, s, r)
    assert s["connected"] == r["connected"], (repo, s, r)
    assert round(100 * s["isolated_share"], 1) == r["isolated"], (repo, s, r)
print("strata_table.json matches recomputation:", len(shipped), "repos")

strata_table.json matches recomputation: 8 repos


## Summary
Every number in the paper's §4 is reproduced above from public artifacts, with no
GPU and no model run. Run order: `scripts/mine_compact.py` →
`scripts/build_compact_graphs.py` → `scripts/stratify.py` (both strata) →
`scripts/extract_tests.py` → `scripts/merge_gold_structure.py` → this notebook.